## Exercițiul 1: Prognoza meteo orară — temperaturi lipsă

Folosim [Open-Meteo](https://open-meteo.com/) pentru prognoza orară de temperatură. Datele reale de la API sunt complete, așa că **simulăm** câteva calluri cu date lipsă — exact scenariul întâlnit des în practică (un senzor care face skip la o citire).

**Cerințe:**
1. Identificați câte valori lipsesc și la ce ore, folosind `.isna()`.
2. Calculați media temperaturilor existente.
3. Completați valorile lipsă cu media (`.fillna()`).
4. Verificați (cu `.isna().sum()`) că nu mai există valori lipsă după completare.

In [ ]:
import requests
import pandas as pd
import numpy as np

raspuns = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 44.43, "longitude": 26.10, "hourly": "temperature_2m", "forecast_days": 1},
)
date_meteo = raspuns.json()

ore = pd.to_datetime(date_meteo["hourly"]["time"])
temperaturi = pd.Series(date_meteo["hourly"]["temperature_2m"], index=ore, name="temperatura")

# datele de la API sunt complete -> simulam cateva valori lipsa (senzor care "sare" o citire)
np.random.seed(42)
pozitii_lipsa = np.random.choice(temperaturi.index, size=4, replace=False)
temperaturi.loc[pozitii_lipsa] = None

print(temperaturi)


## Exercițiul 2: Utilizatori — coordonate GPS 

Folosim [JSONPlaceholder](https://jsonplaceholder.typicode.com/) pentru o listă de utilizatori. **Particularitate reală a acestui API**: coordonatele GPS (`address.geo.lat`) vin ca **text**, nu ca număr — o curățare de tip pe care trebuie să o faceți voi.

**Cerințe:**
1. Convertiți `Series`-ul de latitudini din `string` în `float` (`.astype(float)`).
2. Identificați utilizatorii cu coordonate lipsă (`.isna()`).
3. Completați valorile lipsă cu media latitudinilor existente.
4. Sortați rezultatul final după valoare (`.sort_values()`).

In [ ]:
import requests
import pandas as pd
import numpy as np

raspuns = requests.get("https://jsonplaceholder.typicode.com/users")
utilizatori = raspuns.json()

latitudini_brute = {u["username"]: u["address"]["geo"]["lat"] for u in utilizatori}
latitudine_series = pd.Series(latitudini_brute, name="latitudine")

print(latitudine_series)
print("\ndtype:", latitudine_series.dtype)   # object -> inca text, nu numere

# simulam cateva coordonate lipsa (in date reale, unele inregistrari pot avea campuri goale)
np.random.seed(7)
pozitii_lipsa = np.random.choice(latitudine_series.index, size=3, replace=False)
latitudine_series.loc[pozitii_lipsa] = None


## Exercițiul 3 : Planeta de origine — valori lipsă în date de tip text

Folosim [Rick and Morty API](https://rickandmortyapi.com/) — de data asta, câteva personaje au originea `"unknown"`. E, practic, o **valoare lipsă reală**, doar că API-ul o scrie ca text în loc de `null`.

**Cerințe:**
1. Înlocuiți `"unknown"` cu o valoare lipsă reală pentru pandas (`.replace("unknown", np.nan)`), ca să poată fi detectată cu `.isna()`.
2. Identificați câte și care personaje au originea necunoscută.
3. **De ce NU putem completa aici cu media, ca la exercițiile anterioare?** (e o coloană de text, nu de numere) — completați în schimb valorile lipsă cu cea mai frecventă origine din date (*modul* — `.mode()`), nu cu media.
4. Bonus: folosind accesorul `.str`, numărați câte origini conțin cuvântul `"Earth"`.

In [ ]:
import requests
import pandas as pd
import numpy as np

raspuns = requests.get("https://rickandmortyapi.com/api/character", params={"page": 1})
personaje = raspuns.json()["results"]

origine_series = pd.Series(
    {p["name"]: p["origin"]["name"] for p in personaje},
    name="origine",
)
print(origine_series)
# observati: cateva valori sunt literalmente textul "unknown" -> placeholder-ul folosit de API
# pentru "nu stim", deci practic o valoare LIPSA, doar ca nu apare ca None/NaN


## Exercițiul 4 : Curățarea datelor calendaristice + accesorul `.dt`

Fiecare personaj din Rick and Morty API are un câmp `created` (data la care a fost adăugat în baza de date). Simulăm **o singură** înregistrare coruptă — realist, un caz în care un câmp API vine formatat greșit.

**Cerințe:**
1. Convertiți `Series`-ul în `datetime` cu `pd.to_datetime(..., errors="coerce")` — parametrul `errors="coerce"` face ca orice valoare care nu poate fi convertită să devină `NaT`, în loc să arunce eroare.
2. Identificați rândul (rândurile) cu `NaT`.
3. Discutați și alegeți o strategie de completare: `.ffill()` (ia ultima dată validă anterioară) sau `.dropna()` (elimină rândul) — care are mai mult sens aici și de ce?
4. Extrageți anul fiecărei date cu accesorul **`.dt`** (`.dt.year`) și afișați un `.value_counts()` pe ani.
5. Sortați personajele cronologic după data creării.

In [ ]:
import requests
import pandas as pd

raspuns = requests.get("https://rickandmortyapi.com/api/character", params={"page": 1})
personaje = raspuns.json()["results"]

date_creare_brute = {p["name"]: p["created"] for p in personaje}

# simulam o singura inregistrare "stricata" (un caz real: un camp corupt/gresit formatat)
o_cheie = list(date_creare_brute.keys())[5]
date_creare_brute[o_cheie] = "data necunoscuta"

date_creare_series = pd.Series(date_creare_brute, name="data_creare")
print(date_creare_series)


## Exercițiul 5: Temperatură + precipitații — analiză combinată

Extragem **două** serii meteo în paralel (temperatură și precipitații, pe 2 zile), fiecare cu propriile valori lipsă simulate independent — ca doi senzori diferiți, cu defecțiuni diferite.

**Cerințe:**
1. Identificați și raportați câte valori lipsesc în **fiecare** `Series` în parte.
2. Completați fiecare `Series` cu **propria** medie (`.fillna()`, o dată pentru fiecare serie).
3. Construiți o mască booleană **combinată**: orele în care e frig (`temperatura < 12`) **ȘI** plouă (`precipitatii > 0`) — atenție la `&` și paranteze.
4. Afișați, folosind masca de mai sus, orele care îndeplinesc ambele condiții.
5. Bonus: folosind `.dt.hour` pe index, aflați care oră din zi are, în medie, cele mai multe precipitații (`groupby` pe ora extrasă din index + `.mean()`).

In [ ]:
import requests
import pandas as pd
import numpy as np

raspuns = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 44.43, "longitude": 26.10,
        "hourly": "temperature_2m,precipitation",
        "forecast_days": 2,
    },
)
date_meteo = raspuns.json()

ore = pd.to_datetime(date_meteo["hourly"]["time"])
temperaturi = pd.Series(date_meteo["hourly"]["temperature_2m"], index=ore, name="temperatura")
precipitatii = pd.Series(date_meteo["hourly"]["precipitation"], index=ore, name="precipitatii")

# simulam valori lipsa INDEPENDENTE in fiecare Series (doi "senzori" diferiti, doua defectiuni diferite)
np.random.seed(1)
temperaturi.loc[np.random.choice(temperaturi.index, size=5, replace=False)] = None
np.random.seed(2)
precipitatii.loc[np.random.choice(precipitatii.index, size=5, replace=False)] = None

print(temperaturi.head())
print("\n", precipitatii.head())
